In [ ]:

# 그래드 캠
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import torchvision.transforms as transforms
from PIL import Image
import torch
import torch.nn as nn
import numpy as np
import torchvision.models as models

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image, ImageFile

# Windows 한글 폰트 설정 (맑은 고딕)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
image = Image.open("./images/정약용유적지_12.jpg").convert("RGB")
device = 'cuda'

PLACE_CLASSES = 7
THEME_CLASSES = 5

LABELS_P1 = ["문화재", "문화", "자연", "테마공원", "시장", "거리", "공원"]
LABELS_P2 = ["데이트/로맨틱", "힐링/여유", "액티브/아웃도어", "가족/키즈", "야경/밤감성"]
vgg1 = torch.load("./models/best_vgg_model_epoch_7.pth", map_location=device, weights_only=False)
vgg2 = torch.load("./models/multi_best_vgg_model_epoch_2.pth", map_location=device, weights_only=False)
resnet1 = torch.load("./models/best_place_model_epoch_1.pth", map_location=device, weights_only=False)
resnet2 = torch.load("./models/best_theme_model_epoch_3.pth", map_location=device, weights_only=False)
path_efficientnet1 = "./models/optuna_best_model_efficientnet_part1.pth"
path_efficientnet2 = "./models/optuna_best_model_efficientnet_part2.pth"
path_convnext1 = "./models/best_single_convnext.pth"
path_convnext2 = "./models/best_multi_convnext.pth"

vgg1_in = vgg1.classifier[6].in_features
vgg2_in = vgg2.classifier[6].in_features

resnet1_in = resnet1.fc.in_features
resnet2_in = resnet2.fc.in_features

efficientnet1 = models.efficientnet_b0(weights=None)
efficientnet2 = models.efficientnet_b0(weights=None)
efficientnet1.classifier[1] = nn.Linear(efficientnet1.classifier[1].in_features, PLACE_CLASSES)
efficientnet2.classifier[1] = nn.Linear(efficientnet2.classifier[1].in_features, THEME_CLASSES)
efficientnet1.load_state_dict(torch.load(path_efficientnet1, map_location=device))
efficientnet2.load_state_dict(torch.load(path_efficientnet2, map_location=device))

convnext1 = models.convnext_tiny(weights=None)  
convnext2 = models.convnext_tiny(weights=None)  
convnext1.classifier[2] = nn.Linear(convnext1.classifier[2].in_features, PLACE_CLASSES)
convnext2.classifier[2] = nn.Linear(convnext2.classifier[2].in_features, THEME_CLASSES)
convnext1.load_state_dict(torch.load(path_convnext1, map_location=device))
convnext2.load_state_dict(torch.load(path_convnext2, map_location=device))

vgg1.classifier[6] = nn.Linear(vgg1_in, PLACE_CLASSES)
vgg2.classifier[6] = nn.Linear(vgg2_in, THEME_CLASSES)
                               
resnet1.fc = nn.Linear(resnet1_in, PLACE_CLASSES)
resnet2.fc = nn.Linear(resnet2_in, THEME_CLASSES)
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
from PIL import Image
import matplotlib.pyplot as plt

resnet1.eval()
resnet1.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in resnet1.parameters():
    param.requires_grad = True

pred = resnet1(input_tensor)

pred_class = pred.argmax().item()
pred_label = LABELS_P1[pred_class]

target_layers = resnet1.layer4[-1]
cam = GradCAM(model=resnet1, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"ResNet50 place Grad-CAM (Pred: {pred_label})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

resnet2.eval()
resnet2.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in resnet2.parameters():
    param.requires_grad = True

pred = resnet2(input_tensor)

probs = torch.sigmoid(pred).squeeze()
pred_class = probs.argmax().item()
pred_labels = [LABELS_P2[i] for i, p in enumerate(probs) if p >= 0.5]
if not pred_labels:
    pred_labels = [LABELS_P2[pred_class]]

target_layers = resnet2.layer4[-1]
cam = GradCAM(model=resnet2, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"ResNet50 theme Grad-CAM (Multi-label: {', '.join(pred_labels)})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

efficientnet1.eval()
efficientnet1.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in efficientnet1.parameters():
    param.requires_grad = True

pred = efficientnet1(input_tensor)

pred_class = pred.argmax().item()
pred_label = LABELS_P1[pred_class]

target_layers = efficientnet1.features[-1]
cam = GradCAM(model=efficientnet1, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"EfficientNet-b0 place Grad-CAM (Pred: {pred_label})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

efficientnet2.eval()
efficientnet2.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in efficientnet2.parameters():
    param.requires_grad = True

pred = efficientnet2(input_tensor)

probs = torch.sigmoid(pred).squeeze()
pred_class = probs.argmax().item()
pred_labels = [LABELS_P2[i] for i, p in enumerate(probs) if p >= 0.5]
if not pred_labels:
    pred_labels = [LABELS_P2[pred_class]]

target_layers = efficientnet2.features[-1]
cam = GradCAM(model=efficientnet2, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"EfficientNet-b0 theme Grad-CAM (Multi-label: {', '.join(pred_labels)})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

convnext1.eval()
convnext1.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in convnext1.parameters():
    param.requires_grad = True

pred = convnext1(input_tensor)

pred_class = pred.argmax().item()
pred_label = LABELS_P1[pred_class]

target_layers = convnext1.features[-1]
cam = GradCAM(model=convnext1, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"ConvNeXt-Tiny place Grad-CAM (Pred: {pred_label})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

convnext2.eval()
convnext2.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in convnext2.parameters():
    param.requires_grad = True

pred = convnext2(input_tensor)

probs = torch.sigmoid(pred).squeeze()
pred_class = probs.argmax().item()
pred_labels = [LABELS_P2[i] for i, p in enumerate(probs) if p >= 0.5]
if not pred_labels:
    pred_labels = [LABELS_P2[pred_class]]

target_layers = convnext2.features[-1]
cam = GradCAM(model=convnext2, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"ConvNeXt-Tiny theme Grad-CAM (Multi-label: {', '.join(pred_labels)})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

vgg1.eval()
vgg1.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in vgg1.parameters():
    param.requires_grad = True

pred = vgg1(input_tensor)

pred_class = pred.argmax().item()
pred_label = LABELS_P1[pred_class]

target_layers = vgg1.features[-1]
cam = GradCAM(model=vgg1, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"VGG16 place Grad-CAM (Pred: {pred_label})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

from PIL import Image
import matplotlib.pyplot as plt

vgg2.eval()
vgg2.to("cuda")
img = image

input_tensor = transform(img).unsqueeze(0).to("cuda")

for param in vgg2.parameters():
    param.requires_grad = True

pred = vgg2(input_tensor)

probs = torch.sigmoid(pred).squeeze()
pred_class = probs.argmax().item()
pred_labels = [LABELS_P2[i] for i, p in enumerate(probs) if p >= 0.5]
if not pred_labels:
    pred_labels = [LABELS_P2[pred_class]]

target_layers = vgg2.features[-1]
cam = GradCAM(model=vgg2, target_layers=[target_layers])

grad_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])
rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0

mask = grad_cam[0]  # [H, W], 0~1
overlay = show_cam_on_image(rgb_img, mask, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(f"VGG16 theme Grad-CAM (Multi-label: {', '.join(pred_labels)})", fontsize=14, fontweight="bold")

axes[0].imshow(rgb_img)
axes[0].set_title("1) Original", fontsize=12)
axes[0].axis("off")

m = axes[1].imshow(mask, cmap="jet", vmin=0, vmax=1)
axes[1].set_title("2) CAM Mask", fontsize=12)
axes[1].axis("off")
cbar = fig.colorbar(m, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Activation", rotation=270, labelpad=12)

axes[2].imshow(overlay)
axes[2].set_title("3) Overlay", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()
